# DATA Processing

## MAVEN Magnetometer (MAG) L2
This section performs the ingestion, cleaning, and consolidation of MAVEN Magnetometer (MAG) Level 2 data. The data is processed from raw `.sts` (Standard Time Series) files into a unified, analysis ready format

#### Process Overview
1. **File Ingestion:**
    - Targeted 1 second resolution files (`ss1s`) from the MAVEN MAG instrument
    - Handled the 145-line ASCII header present in `.sts` files to reach the structured data.
2. **Metadata Mapping:**
    - Assigned physical column headers including time (Year, DOY, HMS, msec), Magentic Field Components ($B_x$, $B_y$, $B_z$), and Spacecraft position ($x$, $y$, $z$).
3. **Feature Addition:**
    - Calculated the **Total Magnetic Field Magnitude** ($B_mag$) using the L2-norm of the vector components. $$B_{mag} = \sqrt{B_x^2 + B_y^2 + B_z^2}$$
4. **Consolidation**
    - Merged multiple daily observation files into a single, chronologically sorted global DataFrame.

**Coordinate Systems**

 - **Magnetic Field ($B$):** Measured in nanoTelsa (nT)
 - **Spacecraft Position ($x,y,z$):** Provided in Mars-centric Solar Orbital (MSO) coordinates [km]

In [ ]:
# MAVEN data exploration

# type is STS 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
import xarray as xr


# You will have to input your own data folder path
folder_path = "../../../final-project-data/MAVEN"
print(os.listdir(folder_path))

def read_maven_mag(filepath):
    df = pd.read_csv(
        filepath,
        skiprows=145,
        sep=r'\s+',
        header=None,
        names=[
            'year', 'doy', 'hour', 'min', 'sec', 'msec',
            'dday',                                          
            'Bx', 'By', 'Bz',                             
            'Bflag',                                       
            'x', 'y', 'z',                               
            'posflag',                                    
            'dBx', 'dBy', 'dBz'                         
        ]
    )


    df['datetime'] = pd.to_datetime(df['year'], format='%Y') + \
                    pd.to_timedelta(df['doy'] - 1, unit='D') + \
                    pd.to_timedelta(df['hour'], unit='h') + \
                    pd.to_timedelta(df['min'], unit='m') + \
                    pd.to_timedelta(df['sec'], unit='s') + \
                    pd.to_timedelta(df['msec'], unit='ms')


    return df



files_1s = sorted(glob.glob(os.path.join(folder_path, "*ss1s*.sts")))
print(f"Reading {len(files_1s)} files...")

df_all = pd.concat([read_maven_mag(f) for f in files_1s], ignore_index=True)
df_all = df_all.sort_values('datetime').reset_index(drop=True)
df_all['Bmag'] = np.sqrt(df_all['Bx']**2 + df_all['By']**2 + df_all['Bz']**2)

print(f"Total rows: {len(df_all)}")
print(df_all[['datetime', 'Bmag','Bx', 'By', 'Bz', 'x', 'y', 'z']].head())



['acknowledgement.txt', 'info.txt', 'MAVEN', 'mvn_mag_l2_2017246ss1s_20170903_v01_r01.sts', 'mvn_mag_l2_2017246ss_20170903_v01_r01.sts', 'mvn_mag_l2_2017247ss1s_20170904_v01_r01.sts', 'mvn_mag_l2_2017247ss_20170904_v01_r01.sts', 'mvn_mag_l2_2017248ss1s_20170905_v01_r01.sts', 'mvn_mag_l2_2017248ss_20170905_v01_r01.sts', 'mvn_mag_l2_2017249ss1s_20170906_v01_r01.sts', 'mvn_mag_l2_2017249ss_20170906_v01_r01.sts', 'mvn_mag_l2_2017250ss1s_20170907_v01_r01.sts', 'mvn_mag_l2_2017250ss_20170907_v01_r01.sts', 'mvn_mag_l2_2017251ss1s_20170908_v01_r01.sts', 'mvn_mag_l2_2017251ss_20170908_v01_r01.sts', 'mvn_mag_l2_2017252ss1s_20170909_v01_r01.sts', 'mvn_mag_l2_2017252ss_20170909_v01_r01.sts', 'mvn_mag_l2_2017253ss1s_20170910_v01_r01.sts', 'mvn_mag_l2_2017253ss_20170910_v01_r01.sts', 'mvn_mag_l2_2017254ss1s_20170911_v01_r01.sts', 'mvn_mag_l2_2017254ss_20170911_v01_r01.sts', 'mvn_mag_l2_2017255ss1s_20170912_v01_r01.sts', 'mvn_mag_l2_2017255ss_20170912_v01_r01.sts', 'mvn_mag_l2_2017256ss1s_20170913_v0

#### Exporting Global DataFrame as NetCDF file

In [3]:
# 1. Set datetime as the index so it becomes a coordinate in the NetCDF
df_to_export = df_all.set_index('datetime')

# 2. Convert the Pandas DataFrame to an xarray Dataset
ds = xr.Dataset.from_dataframe(df_to_export)

# 3. Add Metadata (Optional but highly recommended for research)
ds.attrs['description'] = 'MAVEN MAG Level 2 cleaned data'
ds.attrs['units'] = 'nT for B-field, km for position'
ds.attrs['coordinate_system'] = 'MSO'

# 4. Export to NetCDF
output_filename = "MAVEN/MAVEN_MAG_Sept2017_global.nc"
ds.to_netcdf(output_filename)

print(f"Data successfully exported to {output_filename}")

Data successfully exported to MAVEN/MAVEN_MAG_Sept2017_global.nc
